In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/akhilbs/cloud-workload/Workload_dataset.csv


In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Load dataset
df = pd.read_csv('/kaggle/input/datasets/akhilbs/cloud-workload/Workload_dataset.csv')

# Display basic info
print(df.head())
print(df.columns)

         Date  CPU cores  CPU capacity provisioned [MHZ]  CPU usage [MHZ]  \
0  12/19/2008          4                     11703.99824     10434.114430   
1  12/20/2008          4                     11703.99824     10913.978360   
2  12/21/2008          4                     11703.99824     10477.029090   
3  12/22/2008          4                     11703.99824      6485.965691   
4  12/23/2008          4                     11703.99824        76.075989   

   Memory usage [GB]  Memory capacity provisioned [KB]  \
0           8.947846                          67108864   
1           6.352970                          67108864   
2          20.937963                          67108864   
3          11.050593                          67108864   
4           0.000000                          67108864   

   Disk read throughput [KB/s]  Disk write throughput [KB/s]  \
0                     2.533333                  19974.933330   
1                     4.466667                  15553.733330

In [4]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('/kaggle/input/datasets/akhilbs/cloud-workload/Workload_dataset.csv')

# Drop missing values
df = df.dropna()

# ----- Feature Engineering -----

# CPU usage ratio
df['cpu_util'] = df['CPU usage [MHZ]'] / df['CPU capacity provisioned [MHZ]']

# Memory usage ratio (convert KB to GB)
df['mem_util'] = df['Memory usage [KB]'] / df['Memory capacity provisioned [KB]']

# Instances (already given)
df['instances'] = df['Count']

# Request rate proxy (network traffic)
df['request_rate'] = (
    df['Network received throughput [KB/s]'] +
    df['Network transmitted throughput [KB/s]']
)

# Keep only required features
features = ['cpu_util', 'mem_util', 'instances', 'request_rate']
df = df[features]

print(df.head())

   cpu_util  mem_util  instances  request_rate
0  0.891500  0.133333          1    559.600000
1  0.932500  0.094667          1      2.066667
2  0.895167  0.312000          1    239.400000
3  0.554167  0.164667          1    741.400000
4  0.006500  0.000000          1      1.000000


**distribution of instances (Count column).**

In [17]:
instance_counts = df['instances'].value_counts().sort_index()
print(instance_counts)

instances
1    2878
2       1
Name: count, dtype: int64


In [18]:
for inst, count in instance_counts.items():
    print(f"instances = {inst} → {count} rows")

instances = 1 → 2878 rows
instances = 2 → 1 rows


**Data manipulation begin**

In [19]:
import numpy as np

n = len(df)

# Create equal distribution
instances_new = np.random.choice([1, 2, 3], size=n)

df['instances'] = instances_new

print(df['instances'].value_counts())

instances
2    986
3    959
1    934
Name: count, dtype: int64


In [20]:
instances_new = np.random.choice(
    [1, 2, 3],
    size=len(df),
    p=[0.5, 0.3, 0.2]
)

df['instances'] = instances_new

print(df['instances'].value_counts())

instances
1    1437
2     878
3     564
Name: count, dtype: int64


In [24]:
df = df.sample(frac=1).reset_index(drop=True)

In [25]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)

states = df_scaled.values

print("States shape:", states.shape)

States shape: (2879, 4)


In [26]:
import random
import numpy as np

# Reinitialize Q-table (VERY IMPORTANT)
Q = np.zeros((len(states), 3))

# Hyperparameters
alpha = 0.1
gamma = 0.9
epsilon = 1.0
epsilon_decay = 0.995
epsilon_min = 0.01
episodes = 200   # increased for better learning

for episode in range(episodes):
    total_reward = 0

    for i in range(len(states) - 1):
        state = states[i]

        # Epsilon-greedy
        if random.uniform(0, 1) < epsilon:
            action = random.randint(0, 2)
        else:
            action = np.argmax(Q[i])

        reward = get_reward(state, action)

        next_state = i + 1

        # Q-update
        Q[i, action] += alpha * (
            reward + gamma * np.max(Q[next_state]) - Q[i, action]
        )

        total_reward += reward

    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    if episode % 10 == 0:
        print(f"Episode {episode}: Reward = {total_reward:.2f}, Epsilon = {epsilon:.3f}")

Episode 0: Reward = -1576.75, Epsilon = 0.995
Episode 10: Reward = -992.75, Epsilon = 0.946
Episode 20: Reward = -840.75, Epsilon = 0.900
Episode 30: Reward = -511.75, Epsilon = 0.856
Episode 40: Reward = 118.25, Epsilon = 0.814
Episode 50: Reward = 535.25, Epsilon = 0.774
Episode 60: Reward = 841.25, Epsilon = 0.737
Episode 70: Reward = 1263.25, Epsilon = 0.701
Episode 80: Reward = 1531.25, Epsilon = 0.666
Episode 90: Reward = 1908.25, Epsilon = 0.634
Episode 100: Reward = 2227.25, Epsilon = 0.603
Episode 110: Reward = 2360.25, Epsilon = 0.573
Episode 120: Reward = 2576.25, Epsilon = 0.545
Episode 130: Reward = 3023.25, Epsilon = 0.519
Episode 140: Reward = 3304.25, Epsilon = 0.493
Episode 150: Reward = 3465.25, Epsilon = 0.469
Episode 160: Reward = 3500.25, Epsilon = 0.446
Episode 170: Reward = 3856.25, Epsilon = 0.424
Episode 180: Reward = 4109.25, Epsilon = 0.404
Episode 190: Reward = 4164.25, Epsilon = 0.384


In [27]:
actions_map = {
    0: "scale_down",
    1: "hold",
    2: "scale_up"
}

policy = np.argmax(Q, axis=1)
policy_actions = [actions_map[a] for a in policy]

In [28]:
from collections import Counter

action_counts = Counter(policy_actions)

for action, count in action_counts.items():
    print(f"{action} → {count}")

scale_down → 2769
hold → 92
scale_up → 18


In [30]:
df_result = df.copy()
df_result['action'] = policy_actions

df_result.head()
print(df_result[['cpu_util', 'mem_util', 'instances', 'action']].head(20))

    cpu_util  mem_util  instances      action
0   0.006500  0.000000          1  scale_down
1   0.006167  0.000000          2  scale_down
2   0.006333  0.003333          1  scale_down
3   0.006833  0.000000          3  scale_down
4   0.897833  0.136000          1        hold
5   0.006500  0.001333          3  scale_down
6   0.005167  0.000000          2  scale_down
7   0.006667  0.000000          1  scale_down
8   0.007167  0.000000          1  scale_down
9   0.005333  0.000000          1  scale_down
10  0.007500  0.000000          2  scale_down
11  0.007167  0.000000          1  scale_down
12  0.007167  0.000000          1  scale_down
13  0.007167  0.000000          1  scale_down
14  0.006333  0.000000          2  scale_down
15  0.006500  0.000000          2  scale_down
16  0.006167  0.000000          1  scale_down
17  0.005000  0.000000          2  scale_down
18  0.005333  0.000000          1  scale_down
19  0.006000  0.000000          3  scale_down


In [31]:
import json

rl_data = []

for i in range(len(states)):
    state = states[i]

    # Get action from learned policy
    action_idx = np.argmax(Q[i])
    
    action_map = {
        0: "scale_down",
        1: "hold",
        2: "scale_up"
    }
    
    action = action_map[action_idx]

    # Compute reward
    reward = get_reward(state, action_idx)

    entry = {
        "state": {
            "cpu_util": float(state[0]),
            "mem_util": float(state[1]),
            "instances": float(state[2]),
            "request_rate": float(state[3])
        },
        "action": action,
        "reward": float(reward)
    }

    rl_data.append(entry)

# Save JSON
with open('/kaggle/working/rl_dataset.json', 'w') as f:
    json.dump(rl_data, f, indent=4)

print("JSON file created: /kaggle/working/rl_dataset.json")

JSON file created: /kaggle/working/rl_dataset.json
